# Chapter 5 — RoPE, SwiGLU, RMSNorm, and KV cache
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch05_modern_llm_blocks.ipynb)

Implements the modern LLM components introduced in chapter 5 and verifies them with small T4-friendly tensors.

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## 1. Rotary Position Embedding (RoPE)

In [ ]:
def rope(x):
    B,H,T,D=x.shape; assert D%2==0
    pos=torch.arange(T,device=x.device).float()[:,None]
    inv=1.0/(10000**(torch.arange(0,D,2,device=x.device).float()/D))
    ang=pos*inv; cos=ang.cos()[None,None]; sin=ang.sin()[None,None]
    a,b=x[...,0::2],x[...,1::2]
    out=torch.empty_like(x)
    out[...,0::2]=a*cos-b*sin; out[...,1::2]=a*sin+b*cos
    return out
x=torch.randn(2,4,32,64,device=device); print(rope(x).shape)

## 2. SwiGLU and RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self,d,eps=1e-6): super().__init__(); self.w=nn.Parameter(torch.ones(d)); self.eps=eps
    def forward(self,x): return self.w*x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+self.eps)

class SwiGLU(nn.Module):
    def __init__(self,d,hidden): super().__init__(); self.g=nn.Linear(d,hidden,bias=False); self.u=nn.Linear(d,hidden,bias=False); self.o=nn.Linear(hidden,d,bias=False)
    def forward(self,x): return self.o(F.silu(self.g(x))*self.u(x))

z=torch.randn(4,64,128,device=device); print(SwiGLU(128,352).to(device)(RMSNorm(128).to(device)(z)).shape)

## 3. RoPE attention

In [ ]:
class ModernAttention(nn.Module):
    def __init__(self,d=128,h=4):
        super().__init__(); self.h=h; self.hd=d//h; self.qkv=nn.Linear(d,3*d,bias=False); self.o=nn.Linear(d,d,bias=False)
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.qkv(x).chunk(3,-1)
        reshape=lambda a:a.view(B,T,self.h,self.hd).transpose(1,2)
        q,k,v=map(reshape,(q,k,v)); q,k=rope(q),rope(k)
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        return self.o(y.transpose(1,2).contiguous().view(B,T,C))
print(ModernAttention().to(device)(z).shape)

## 4. KV-cache idea
During autoregressive decoding, previous keys and values do not change. Cache them and append only the new token's K/V instead of recomputing the entire prefix.

In [ ]:
def append_kv(cache_k,cache_v,new_k,new_v):
    if cache_k is None: return new_k,new_v
    return torch.cat([cache_k,new_k],dim=-2),torch.cat([cache_v,new_v],dim=-2)
ck=cv=None
for t in range(5):
    nk=torch.randn(1,4,1,32,device=device); nv=torch.randn_like(nk)
    ck,cv=append_kv(ck,cv,nk,nv)
    print(t,ck.shape)

## T4 note
All chapter-5 components are comfortably testable on a T4. Increase sequence length after correctness checks to observe KV-cache memory/speed tradeoffs.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch05